# CIFAR-10 CNN: Tucker-2 圧縮後の fine-tuning

`02_rank_sweep.ipynb` が Pareto / knee から自動選択した

- aggressive
- balanced
- conservative

の 3 候補だけを fine-tuning する。

ここでの目的は **圧縮 → 精度低下 → fine-tuning で回復** を、
圧縮強度の異なる 3 点で公平に比較すること。

rank 探索はこの Notebook では行わない。

## 2026-09-12 保存成果物監査への対応

`finetuning_comparison.csv` の `weight_relative_error` は02で測った圧縮直後（FT前）の参照値をbefore/after両行に載せる列。after行もFT後の再計算値ではない。今回の修正は列の意味の明記だけで、学習・結果値の更新はしていない。

保存出力の整合性と、現行コードのRun All成功は別の確認である。未実行の追加処理があるセルはexecution_countを空にし、既存outputは過去記録として保持した。


## 1. baseline・DataLoader・共通処理を準備する

比較条件は `02_rank_sweep.ipynb` / SVD corrected 版と同じにする。

- split: train 40k / val_early_stop 5k / val_rank 5k / test 10k
- baseline: `02_svd_global_compression_using_src_corrected/cifar10_cnn_baseline.pt`
- Early Stopping 用 validation は `validation_loader_early_stop`
- test は最終評価のみ


In [3]:
from __future__ import annotations

from pathlib import Path
import copy
import sys

print("import: numpy / pandas / matplotlib", flush=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("import: torch", flush=True)
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

for _candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _src = _candidate / "src"
    if (_src / "nn_compression").is_dir():
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break

print("import: nn_compression", flush=True)
from nn_compression.compression import build_tucker2_conv
from nn_compression.datasets import shuffled_index_splits
from nn_compression.metrics import (
    count_parameters,
    parameters_reduction,
)
from nn_compression.models import CIFAR10CNN
from nn_compression.training import (
    evaluate,
    fit_with_early_stopping,
    non_shuffling_loader,
)
from nn_compression.utils import (
    find_project_root,
    get_experiment_dirs,
    set_seed,
)

project_root = find_project_root(Path.cwd())

METHOD_NAME = "20_tucker"
CASE_NAME = "10_cifar10_cnn"
EXPERIMENT_NAME = "03_finetuning"

data_dir, models_dir, results_dir = get_experiment_dirs(
    project_root,
    METHOD_NAME,
    CASE_NAME,
    EXPERIMENT_NAME,
)

# 02 の出力（rank 選択結果）
RANK_SWEEP_RESULTS_DIR = (
    project_root / "results" / METHOD_NAME / CASE_NAME / "02_rank_sweep"
)
SELECTED_RANKS_CSV = RANK_SWEEP_RESULTS_DIR / "selected_rank_settings.csv"

MODEL_PATH = (
    project_root
    / "models/10_svd/40_cifar10_cnn/02_svd_global_compression_using_src_corrected"
    / "cifar10_cnn_baseline.pt"
)

SEED = 0
set_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("project_root:", project_root)
print("results_dir:", results_dir)
print("models_dir:", models_dir)
print("selected ranks csv:", SELECTED_RANKS_CSV)
print("baseline:", MODEL_PATH)
print("device:", device)
print("PyTorch:", torch.__version__)


import: numpy / pandas / matplotlib
import: torch
import: nn_compression
project_root: D:\dev\nn-compression-svd-dmrg
results_dir: D:\dev\nn-compression-svd-dmrg\results\20_tucker\10_cifar10_cnn\03_finetuning
models_dir: D:\dev\nn-compression-svd-dmrg\models\20_tucker\10_cifar10_cnn\03_finetuning
selected ranks csv: D:\dev\nn-compression-svd-dmrg\results\20_tucker\10_cifar10_cnn\02_rank_sweep\selected_rank_settings.csv
baseline: D:\dev\nn-compression-svd-dmrg\models\10_svd\40_cifar10_cnn\02_svd_global_compression_using_src_corrected\cifar10_cnn_baseline.pt
device: cuda
PyTorch: 2.11.0+cu128


In [4]:
# CIFAR-10 前処理（02 / SVD corrected と同じ）
train_transform = transforms.Compose(
    [
        transforms.RandomCrop(size=32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
)
evaluation_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
)

full_train_augmented = datasets.CIFAR10(
    root=data_dir,
    train=True,
    download=False,
    transform=train_transform,
)
full_train_evaluation = datasets.CIFAR10(
    root=data_dir,
    train=True,
    download=False,
    transform=evaluation_transform,
)
test_dataset = datasets.CIFAR10(
    root=data_dir,
    train=False,
    download=False,
    transform=evaluation_transform,
)

TRAIN_SIZE = 40_000
VALIDATION_SIZE = 5_000
train_indices, validation_indices_early_stop, validation_indices_rank = (
    shuffled_index_splits(
        len(full_train_augmented),
        (TRAIN_SIZE, VALIDATION_SIZE, VALIDATION_SIZE),
        seed=SEED,
    )
)

train_dataset = Subset(full_train_augmented, indices=train_indices)
validation_dataset_early_stop = Subset(
    full_train_evaluation,
    indices=validation_indices_early_stop,
)
validation_dataset_rank = Subset(
    full_train_evaluation,
    indices=validation_indices_rank,
)

BATCH_SIZE = 256
NUM_WORKERS = 0
loader_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    generator=loader_generator,
)
train_eval_loader = non_shuffling_loader(train_loader)
validation_loader_early_stop = DataLoader(
    validation_dataset_early_stop,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
validation_loader_rank = DataLoader(
    validation_dataset_rank,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print(f"train: {len(train_dataset):,}")
print(f"validation_early_stop: {len(validation_dataset_early_stop):,}")
print(f"validation_rank: {len(validation_dataset_rank):,}")
print(f"test: {len(test_dataset):,}")


train: 40,000
validation_early_stop: 5,000
validation_rank: 5,000
test: 10,000


## 2. 02 で選んだ 3 rank を読み込む

`selected_rank_settings.csv` が無い場合は、デフォルト rank へ落とさない。
先に `02_rank_sweep.ipynb` を最後まで実行する。


In [5]:
if not SELECTED_RANKS_CSV.is_file():
    raise FileNotFoundError(
        "selected_rank_settings.csv が見つかりません。\n"
        f"expected: {SELECTED_RANKS_CSV}\n"
        "先に notebooks/20_tucker/10_cifar10_cnn/02_rank_sweep.ipynb を "
        "Restart & Run All 相当で実行し、Pareto / knee 選択まで完了してください。"
    )

selected_rank_settings_df = pd.read_csv(SELECTED_RANKS_CSV)
required_roles = {"aggressive", "balanced", "conservative"}
got_roles = set(selected_rank_settings_df["role"].tolist())
if got_roles != required_roles:
    raise ValueError(
        f"selected_rank_settings.csv の role が不正です: {got_roles} "
        f"(expected {required_roles})"
    )

role_order = {"aggressive": 0, "balanced": 1, "conservative": 2}
selected_rank_settings_df = (
    selected_rank_settings_df.assign(
        _role_order=selected_rank_settings_df["role"].map(role_order)
    )
    .sort_values("_role_order")
    .drop(columns="_role_order")
    .reset_index(drop=True)
)

print("Loaded selected ranks from 02:")
for _, row in selected_rank_settings_df.iterrows():
    print(
        f"  {row['role']:12s}: rank_out={int(row['rank_out'])}, "
        f"rank_in={int(row['rank_in'])}"
    )
selected_rank_settings_df


Loaded selected ranks from 02:
  aggressive  : rank_out=16, rank_in=24
  balanced    : rank_out=32, rank_in=16
  conservative: rank_out=32, rank_in=24


,role,rank_out,rank_in,parameters,parameters_reduction,validation_loss,validation_acc,accuracy_drop,conv2_macs_reduction,all_macs_reduction,weight_relative_error
0,aggressive,16.0,24.0,115658.0,0.102327,1.334486,0.5162,0.2182,0.715278,0.325869,0.565492
1,balanced,32.0,16.0,117578.0,0.087425,1.057830,0.6280,0.1064,0.611111,0.278412,0.449042
2,conservative,32.0,24.0,120138.0,0.067556,0.977474,0.6532,0.0812,0.472222,0.215137,0.377868


## 3. baseline を評価する

圧縮前モデルの validation / test / parameters を 1 回だけ測る。


In [6]:
criterion = nn.CrossEntropyLoss()

baseline_model = CIFAR10CNN().to(device)
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint
baseline_model.load_state_dict(state_dict)
baseline_model.eval()

baseline_params = count_parameters(baseline_model)
baseline_val_loss, baseline_val_acc = evaluate(
    baseline_model,
    validation_loader_rank,
    criterion,
    device,
)
baseline_test_loss, baseline_test_acc = evaluate(
    baseline_model,
    test_loader,
    criterion,
    device,
)

print(f"baseline params: {baseline_params}")
print(f"baseline val  loss={baseline_val_loss:.4f} acc={baseline_val_acc:.4f}")
print(f"baseline test loss={baseline_test_loss:.4f} acc={baseline_test_acc:.4f}")


baseline params: 128842
baseline val  loss=0.7602 acc=0.7344
baseline test loss=0.7556 acc=0.7327


## 4. fine-tuning 条件

SVD corrected 版の Fine-tuning に合わせる（3 候補で共通）。

| 項目 | 値 |
|---|---|
| optimizer | Adam |
| learning rate | `3e-4`（baseline 学習の 1e-3 より一段下げる） |
| max epochs | 30 |
| patience | 3 |
| min_delta | 1e-4 |
| seed | 0（各候補の開始時に `set_seed` + DataLoader Generator をリセット） |
| Early Stopping validation | `validation_loader_early_stop` |
| `reevaluate_train` | False |

候補間の差が初期乱数ではなく rank 差になるようにする。


In [7]:
LEARNING_RATE = 3e-4
MAX_EPOCHS = 30
PATIENCE = 3
MIN_DELTA = 1e-4

print(
    f"FT config: Adam lr={LEARNING_RATE}, max_epochs={MAX_EPOCHS}, "
    f"patience={PATIENCE}, min_delta={MIN_DELTA}, seed={SEED}"
)


FT config: Adam lr=0.0003, max_epochs=30, patience=3, min_delta=0.0001, seed=0


## 5. 3 候補を圧縮 → 直後評価 → fine-tuning → 再評価

各候補は必ず baseline から `deepcopy` し直す。
前候補の fine-tuned 重みを引き継がない。
対象層は **conv2 のみ**。


In [ ]:
# weight_relative_error は02の圧縮直後の参照値をbefore/after両行へ載せる。
# after行でもFT後に再計算した誤差ではない（artifact_provenance.jsonにも定義を記録）。
comparison_rows = []
histories = {}
finetuned_models = {}

# baseline 行
comparison_rows.append(
    {
        "stage": "baseline",
        "role": "baseline",
        "rank_out": np.nan,
        "rank_in": np.nan,
        "validation_loss": baseline_val_loss,
        "validation_acc": baseline_val_acc,
        "test_loss": baseline_test_loss,
        "test_acc": baseline_test_acc,
        "accuracy_drop": 0.0,
        "parameters": baseline_params,
        "parameters_reduction": 0.0,
        "conv2_macs_reduction": np.nan,
        "all_macs_reduction": np.nan,
        "weight_relative_error": np.nan,
        "best_epoch": np.nan,
    }
)

for _, setting in selected_rank_settings_df.iterrows():
    role = str(setting["role"])
    rank_out = int(setting["rank_out"])
    rank_in = int(setting["rank_in"])

    print(f"\n=== {role}: rank_out={rank_out}, rank_in={rank_in} ===")

    # --- A. 圧縮直後（baseline から新規） ---
    compressed_model = copy.deepcopy(baseline_model)
    compressed_model.conv2 = build_tucker2_conv(
        compressed_model.conv2,
        rank_out=rank_out,
        rank_in=rank_in,
    ).to(device)

    pre_params = count_parameters(compressed_model)
    pre_val_loss, pre_val_acc = evaluate(
        compressed_model,
        validation_loader_rank,
        criterion,
        device,
    )
    pre_test_loss, pre_test_acc = evaluate(
        compressed_model,
        test_loader,
        criterion,
        device,
    )
    comparison_rows.append(
        {
            "stage": f"{role}_before_finetuning",
            "role": role,
            "rank_out": rank_out,
            "rank_in": rank_in,
            "validation_loss": pre_val_loss,
            "validation_acc": pre_val_acc,
            "test_loss": pre_test_loss,
            "test_acc": pre_test_acc,
            "accuracy_drop": baseline_val_acc - pre_val_acc,
            "parameters": pre_params,
            "parameters_reduction": parameters_reduction(
                baseline_model, compressed_model
            ),
            "conv2_macs_reduction": setting.get("conv2_macs_reduction", np.nan),
            "all_macs_reduction": setting.get("all_macs_reduction", np.nan),
            "weight_relative_error": setting.get("weight_relative_error", np.nan),
            "best_epoch": np.nan,
        }
    )
    print(
        f"  before FT: val_acc={pre_val_acc:.4f} test_acc={pre_test_acc:.4f} "
        f"params={pre_params}"
    )

    # --- B. fine-tuning（同じ圧縮モデルを継続学習） ---
    set_seed(SEED)
    loader_generator.manual_seed(SEED)
    optimizer = torch.optim.Adam(
        compressed_model.parameters(),
        lr=LEARNING_RATE,
    )
    fit_result = fit_with_early_stopping(
        model=compressed_model,
        train_loader=train_loader,
        val_loader=validation_loader_early_stop,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        min_delta=MIN_DELTA,
        reevaluate_train=False,
        log_every_epoch=True,
    )

    ft_model = fit_result["model"]
    post_params = count_parameters(ft_model)
    post_val_loss, post_val_acc = evaluate(
        ft_model,
        validation_loader_rank,
        criterion,
        device,
    )
    post_test_loss, post_test_acc = evaluate(
        ft_model,
        test_loader,
        criterion,
        device,
    )
    comparison_rows.append(
        {
            "stage": f"{role}_after_finetuning",
            "role": role,
            "rank_out": rank_out,
            "rank_in": rank_in,
            "validation_loss": post_val_loss,
            "validation_acc": post_val_acc,
            "test_loss": post_test_loss,
            "test_acc": post_test_acc,
            "accuracy_drop": baseline_val_acc - post_val_acc,
            "parameters": post_params,
            "parameters_reduction": parameters_reduction(baseline_model, ft_model),
            "conv2_macs_reduction": setting.get("conv2_macs_reduction", np.nan),
            "all_macs_reduction": setting.get("all_macs_reduction", np.nan),
            "weight_relative_error": setting.get("weight_relative_error", np.nan),
            "best_epoch": fit_result["best_epoch"],
        }
    )
    print(
        f"  after FT:  val_acc={post_val_acc:.4f} test_acc={post_test_acc:.4f} "
        f"best_epoch={fit_result['best_epoch']}"
    )

    histories[role] = pd.DataFrame(fit_result["history"])
    finetuned_models[role] = {
        "model": ft_model,
        "rank_out": rank_out,
        "rank_in": rank_in,
    }

df_comparison = pd.DataFrame(comparison_rows)
df_comparison



=== aggressive: rank_out=16, rank_in=24 ===
  before FT: val_acc=0.5162 test_acc=0.5297 params=115658
epoch=01 train_loss=0.8672 train_acc=0.6961 val_loss=0.8518 val_acc=0.7040
epoch=02 train_loss=0.8008 train_acc=0.7197 val_loss=0.8213 val_acc=0.7128
epoch=03 train_loss=0.7901 train_acc=0.7253 val_loss=0.8119 val_acc=0.7174
epoch=04 train_loss=0.7728 train_acc=0.7303 val_loss=0.8163 val_acc=0.7156
epoch=05 train_loss=0.7617 train_acc=0.7347 val_loss=0.8134 val_acc=0.7202
epoch=06 train_loss=0.7573 train_acc=0.7368 val_loss=0.7880 val_acc=0.7288
epoch=07 train_loss=0.7533 train_acc=0.7345 val_loss=0.8004 val_acc=0.7264
epoch=08 train_loss=0.7416 train_acc=0.7407 val_loss=0.7826 val_acc=0.7262
epoch=09 train_loss=0.7439 train_acc=0.7404 val_loss=0.7730 val_acc=0.7334
epoch=10 train_loss=0.7342 train_acc=0.7440 val_loss=0.7717 val_acc=0.7286
epoch=11 train_loss=0.7287 train_acc=0.7462 val_loss=0.7732 val_acc=0.7340
epoch=12 train_loss=0.7241 train_acc=0.7453 val_loss=0.7815 val_acc=0.73

,stage,role,rank_out,rank_in,validation_loss,validation_acc,test_loss,test_acc,accuracy_drop,parameters,parameters_reduction,conv2_macs_reduction,all_macs_reduction,weight_relative_error,best_epoch
0,baseline,baseline,NaN,NaN,0.760244,0.7344,0.755618,0.7327,0.0000,128842,0.000000,NaN,NaN,NaN,NaN
1,aggressive_before_finetuning,aggressive,16.0,24.0,1.334486,0.5162,1.305455,0.5297,0.2182,115658,0.102327,0.715278,0.325869,0.565492,NaN
2,aggressive_after_finetuning,aggressive,16.0,24.0,0.729494,0.7462,0.734255,0.7476,-0.0118,115658,0.102327,0.715278,0.325869,0.565492,16.0
3,balanced_before_finetuning,balanced,32.0,16.0,1.057830,0.6280,1.026179,0.6378,0.1064,117578,0.087425,0.611111,0.278412,0.449042,NaN
4,balanced_after_finetuning,balanced,32.0,16.0,0.676505,0.7628,0.676745,0.7682,-0.0284,117578,0.087425,0.611111,0.278412,0.449042,30.0
5,conservative_before_finetuning,conservative,32.0,24.0,0.977474,0.6532,0.950204,0.6612,0.0812,120138,0.067556,0.472222,0.215137,0.377868,NaN
6,conservative_after_finetuning,conservative,32.0,24.0,0.701119,0.7568,0.698856,0.7571,-0.0224,120138,0.067556,0.472222,0.215137,0.377868,19.0


## 6. 学習曲線

各候補の Early Stopping 用 validation 指標（`fit_with_early_stopping` の履歴）。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for role, hist in histories.items():
    axes[0].plot(hist["epoch"], hist["validation_loss"], marker="o", label=role)
    axes[1].plot(hist["epoch"], hist["validation_acc"], marker="o", label=role)

axes[0].set_xlabel("epoch")
axes[0].set_ylabel("validation_loss")
axes[0].set_title("Fine-tuning validation loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("epoch")
axes[1].set_ylabel("validation_acc")
axes[1].set_title("Fine-tuning validation accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curve_path = results_dir / "finetuning_learning_curves.png"
fig.savefig(curve_path, dpi=150, bbox_inches="tight")
print("saved:", curve_path)
plt.show()


## 7. 比較表・モデル・CSV を保存する


In [10]:
results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

comparison_csv_path = results_dir / "finetuning_comparison.csv"
df_comparison.to_csv(comparison_csv_path, index=False)
print("saved:", comparison_csv_path)

for role, hist in histories.items():
    hist_path = results_dir / f"{role}_history.csv"
    hist.to_csv(hist_path, index=False)
    print("saved:", hist_path)

saved_model_paths = {}
for role, info in finetuned_models.items():
    rank_out = info["rank_out"]
    rank_in = info["rank_in"]
    model_path = (
        models_dir
        / f"tucker2_conv2_{role}_r{rank_out}_r{rank_in}_finetuned.pt"
    )
    torch.save(info["model"].state_dict(), model_path)
    saved_model_paths[role] = model_path
    print("saved:", model_path)

print("\n=== comparison ===")
display_cols = [
    "stage",
    "role",
    "rank_out",
    "rank_in",
    "validation_acc",
    "test_acc",
    "accuracy_drop",
    "parameters",
    "parameters_reduction",
    "conv2_macs_reduction",
    "all_macs_reduction",
]
df_comparison[display_cols]


saved: D:\dev\nn-compression-svd-dmrg\results\20_tucker\10_cifar10_cnn\03_finetuning\finetuning_comparison.csv
saved: D:\dev\nn-compression-svd-dmrg\results\20_tucker\10_cifar10_cnn\03_finetuning\aggressive_history.csv
saved: D:\dev\nn-compression-svd-dmrg\results\20_tucker\10_cifar10_cnn\03_finetuning\balanced_history.csv
saved: D:\dev\nn-compression-svd-dmrg\results\20_tucker\10_cifar10_cnn\03_finetuning\conservative_history.csv
saved: D:\dev\nn-compression-svd-dmrg\models\20_tucker\10_cifar10_cnn\03_finetuning\tucker2_conv2_aggressive_r16_r24_finetuned.pt
saved: D:\dev\nn-compression-svd-dmrg\models\20_tucker\10_cifar10_cnn\03_finetuning\tucker2_conv2_balanced_r32_r16_finetuned.pt
saved: D:\dev\nn-compression-svd-dmrg\models\20_tucker\10_cifar10_cnn\03_finetuning\tucker2_conv2_conservative_r32_r24_finetuned.pt

=== comparison ===


,stage,role,rank_out,rank_in,validation_acc,test_acc,accuracy_drop,parameters,parameters_reduction,conv2_macs_reduction,all_macs_reduction
0,baseline,baseline,NaN,NaN,0.7344,0.7327,0.0000,128842,0.000000,NaN,NaN
1,aggressive_before_finetuning,aggressive,16.0,24.0,0.5162,0.5297,0.2182,115658,0.102327,0.715278,0.325869
2,aggressive_after_finetuning,aggressive,16.0,24.0,0.7462,0.7476,-0.0118,115658,0.102327,0.715278,0.325869
3,balanced_before_finetuning,balanced,32.0,16.0,0.6280,0.6378,0.1064,117578,0.087425,0.611111,0.278412
4,balanced_after_finetuning,balanced,32.0,16.0,0.7628,0.7682,-0.0284,117578,0.087425,0.611111,0.278412
5,conservative_before_finetuning,conservative,32.0,24.0,0.6532,0.6612,0.0812,120138,0.067556,0.472222,0.215137
6,conservative_after_finetuning,conservative,32.0,24.0,0.7568,0.7571,-0.0224,120138,0.067556,0.472222,0.215137


## 8. この Notebook の完了条件

1. 02 の selected ranks を読んで 3 候補を fine-tuning できた
2. 圧縮直後と fine-tuning 後の validation / test を比較できた
3. パラメータ削減を維持したまま精度回復を確認できた
4. checkpoint と CSV を method/case/experiment 配下へ保存できた
